# 09 — Target-load comparison and bounded saturation

The E-Perf-1 Target-Load Delivery Summary is target-load evidence; E-Perf-10 is a fixed-duration open-loop saturation diagnostic, not a maximum-throughput claim. Historical eKuiper QoS-0 tails are invalid comparator evidence and are not subtracted. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.

In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import find_canonical_ledger, resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

batch_id=os.environ.get('WAFER_EVAL_BATCH_ID')
try:
    batch=resolve_result_batch('e-perf-10', diagnostic_path=os.environ.get('E_PERF_10_DIR'))
except (FileNotFoundError, RuntimeError, ValueError):
    batch=None
artifacts=[] if batch is None else passed_json(batch,'rate-sweep.json')
raw=pd.DataFrame([{
    'system':value['system'],
    'offered_rate_msg_s':value['offered_rate_msg_s'],
    'run':path.parent.name,
    'achieved_rate_msg_s':value['achieved_rate_msg_s'],
    'p95_ms':value['latency_ns']['p95']/1e6,
    'p99_ms':value['latency_ns']['p99']/1e6,
    'loss_percent':value['loss_percent'],
} for path,value in artifacts])
rows=[]
for system in ['mqtt-loopback','native','wafer','ekuiper']:
    for rate in [500,1000,2000,4000,8000,16000]:
        values=raw[(raw.system == system) & (raw.offered_rate_msg_s == rate)] if not raw.empty else raw
        if values.empty:
            row=pending_record(f'{system} at {rate} messages/second','no passed rate-sweep.json leaf','messages/second, milliseconds, percent')
            row.update({'system':system,'offered_rate_msg_s':rate,'N_runs':0})
            rows.append(row)
        else:
            rows.append({
                'question':f'{system} at {rate} messages/second',
                'status':'READY',
                'system':system,
                'offered_rate_msg_s':rate,
                'N_runs':len(values),
                'median_achieved_rate_msg_s':values['achieved_rate_msg_s'].median(),
                'median_p95_ms':values['p95_ms'].median(),
                'median_p99_ms':values['p99_ms'].median(),
                'median_loss_percent':values['loss_percent'].median(),
                'units':'messages/second, milliseconds, percent',
                'uncertainty':'descriptive only',
                'thesis_evidence':False,
            })
df=pd.DataFrame(rows)
ready=df[df.status=='READY']
print(f"{evidence_label(len(raw), 'messages/second, milliseconds, percent', False)}; independent runs across {len(ready)} fixed-rate conditions; per-condition N shown")
display(df)
if not ready.empty:
    fig, ax=plt.subplots()
    for system, group in ready.groupby('system'):
        ax.plot(group['offered_rate_msg_s'],group['median_p99_ms'],marker='o',label=system)
    ax.set_yscale('log')
    ax.set_xlabel('Offered rate (messages/second)')
    ax.set_ylabel('Median run p99 latency (ms, log scale)')
    ax.set_title('Fixed-duration open-loop sweep (diagnostic; not maximum throughput)')
    ax.legend()

summary=None
summaries=[] if batch is None else passed_json(batch,'rate-sweep-summary.json')
if summaries:
    summary=summaries[0][1]
elif batch_id:
    try:
        summary_path=find_canonical_ledger(batch_id)/'rate-sweep-summary.json'
        if summary_path.is_file():
            summary=json.loads(summary_path.read_text())
    except (FileNotFoundError, ValueError):
        pass
if summary is None:
    display(pd.DataFrame([pending_record('target load versus bounded saturation sweep','rate-sweep-summary.json is unavailable','messages/second')]))
else:
    decision_rows=[]
    for system, value in summary['systems'].items():
        samples=list(value['observed_samples'].values())
        decision_rows.append({
            'system':system,
            'conditions':len(samples),
            'N_runs_per_condition':min(samples) if len(set(samples)) == 1 else str(sorted(set(samples))),
            'target_load_baseline_msg_s':value['baseline_rate_msg_s'],
            'last_good_tested_rate_msg_s':value['last_good_rate_msg_s'],
            'first_bad_tested_rate_msg_s':value['first_bad_rate_msg_s'],
            'highest_tested_rate_msg_s':value['highest_tested_rate_msg_s'],
            'units':'messages/second',
            'uncertainty':'descriptive only',
            'thesis_evidence':False,
            'claim_boundary':'bounded tested rates; not maximum sustainable throughput',
        })
    display(pd.DataFrame(decision_rows))
